In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
# 1. 数据读取与严格对齐
# 读取 train_val 与 test 两个分片的 metadata，并合并
meta_train_val = pd.read_csv('WAYB_WAYC_metadata_train_val(1).csv')
meta_test = pd.read_csv('WAYB_WAYC_metadata_test(1).csv')
meta = pd.concat([meta_train_val, meta_test], ignore_index=True)

# 读取 train_val 与 test 两个分片的蛋白丰度，并合并
protein_train_val = pd.read_csv('WAYB_WAYC_proteome_raw_train_val.csv')
protein_test = pd.read_csv('WAYB_WAYC_proteome_raw_test.csv')
protein = pd.concat([protein_train_val, protein_test], ignore_index=True)

print('meta shape:', meta.shape)
print('protein shape:', protein.shape)

# 将索引设置为 'sample_ID'
meta = meta.set_index('sample_ID')
protein = protein.set_index('sample_ID')

# 取交集，仅保留两边都存在的样本（Inner Join）
common_ids = meta.index.intersection(protein.index)
meta = meta.loc[common_ids]
protein = protein.loc[common_ids]

print(f"对齐后总样本数: {len(common_ids)}")

meta shape: (13412, 15)
protein shape: (13412, 5244)


对齐后总样本数: 13412


In [3]:
# 2. 特征/标签过滤（严格仅在训练集上计算，防止数据泄漏）
train_mask = meta['split_final'] == 'train'
train_protein = protein.loc[train_mask]

# 计算训练集中各蛋白列的缺失率
missing_rate = train_protein.isna().mean()

# 筛选并仅保留缺失率 < 80% 的蛋白列
valid_protein_cols = missing_rate[missing_rate < 0.8].index
print(f"保留的蛋白列数量 (缺失率 < 80%): {len(valid_protein_cols)}")

# 仅保留筛选后的蛋白列
protein_filtered = protein[valid_protein_cols]

保留的蛋白列数量 (缺失率 < 80%): 4422


In [4]:
# 3. 数据转换与 Mask 保留
# 对筛选后的蛋白表达量矩阵执行 np.log2() 变换，NaN 保持为 NaN（缺失 Mask）
protein_log2 = np.log2(protein_filtered)

# 验证 NaN 仍然是 NaN（缺失 Mask 保留）
print(f"原始 NaN 数量: {protein_filtered.isna().sum().sum()}")
print(f"log2 后 NaN 数量: {protein_log2.isna().sum().sum()}")
print(f"NaN 保持一致: {protein_filtered.isna().sum().sum() == protein_log2.isna().sum().sum()}")

原始 NaN 数量: 8974734
log2 后 NaN 数量: 8974734


NaN 保持一致: True


In [5]:
# 4. Baseline 1 均值预测生成
# 训练集的 log2 蛋白矩阵
train_log2 = protein_log2.loc[train_mask]

# 计算训练集中每个蛋白列在非缺失值下的 log2 平均值，得到全局蛋白均值向量
protein_mean_vector = train_log2.mean(axis=0, skipna=True)

# 验证集 (split_final 以 'val' 开头) 的所有样本
val_mask = meta['split_final'].str.startswith('val')
val_log2 = protein_log2.loc[val_mask]

# 构建预测矩阵：每一个验证集样本的所有蛋白预测值都等于该 protein_mean_vector
val_preds = pd.DataFrame(
    np.tile(protein_mean_vector.values, (val_log2.shape[0], 1)),
    index=val_log2.index,
    columns=val_log2.columns
)

print(f"验证集样本数: {val_log2.shape[0]}")
print(f"验证集预测矩阵 shape: {val_preds.shape}")

验证集样本数: 3038
验证集预测矩阵 shape: (3038, 4422)


In [6]:
# 5. 评估指标计算与打印
val_true = val_log2  # 验证集真实的 log2 蛋白表达量矩阵

# 在计算评估指标时，只针对真实值 val_true 中非 NaN（非缺失）的位置进行评估
valid_mask = val_true.notna().values  # numpy 布尔数组

# 提取有效位置的真实值和预测值（1D 数组）
y_true_valid = val_true.values[valid_mask]
y_pred_valid = val_preds.values[valid_mask]

# 计算总体 RMSE (Root Mean Squared Error)
rmse = np.sqrt(np.mean((y_true_valid - y_pred_valid) ** 2))

# 计算全局 R2 Score（只传入非缺失有效值对）
r2 = r2_score(y_true_valid, y_pred_valid)

print(f"===== Baseline 1 (全局蛋白均值基线) 结果 =====")
print(f"对齐后总样本数: {len(common_ids)}")
print(f"保留的蛋白列数量: {len(valid_protein_cols)}")
print(f"验证集样本数: {val_log2.shape[0]}")
print(f"有效评估位置数 (非缺失): {valid_mask.sum()}")
print(f"RMSE: {rmse:.6f}")
print(f"R2 Score: {r2:.6f}")

===== Baseline 1 (全局蛋白均值基线) 结果 =====
对齐后总样本数: 13412
保留的蛋白列数量: 4422
验证集样本数: 3038
有效评估位置数 (非缺失): 11521736
RMSE: 0.938526
R2 Score: 0.885639


In [7]:
# ===== Baseline 2: 带逐级回退机制的 Control Baseline =====
# 1. 定义与提取对照组（Control）样本
# Control 判定：perturbation_no_concentration 属于 DMSO / Water / Quality Control
control_chemicals = ['DMSO', 'Water', 'Quality Control']

# 严格仅使用训练集中的 Control 样本，防止数据泄漏
train_meta = meta.loc[train_mask]
is_control = train_meta['perturbation_no_concentration'].isin(control_chemicals)
train_control_meta = train_meta.loc[is_control]
train_control_log2 = protein_log2.loc[train_control_meta.index]

print(f"训练集 Control 样本数: {len(train_control_meta)}")
print(f"Control 化学物分布:")
print(train_control_meta['perturbation_no_concentration'].value_counts())
print(f"\nControl 涉及的菌株: {sorted(train_control_meta['Strains'].unique())}")

训练集 Control 样本数: 842
Control 化学物分布:
perturbation_no_concentration
Water              383
DMSO               368
Quality Control     91
Name: count, dtype: int64

Control 涉及的菌株: ['BAH', 'CEK', 'CGD', 'DHY210']


In [8]:
# 2. 预计算各回退层级的 Control 均值向量
# Level 1: strain + medium + temperature 严格匹配
ctrl_lvl1 = train_control_log2.groupby(
    [train_control_meta['Strains'], train_control_meta['Medium'], train_control_meta['Temperature']]
).mean()

# Level 2: 同菌株回退
ctrl_lvl2 = train_control_log2.groupby(train_control_meta['Strains']).mean()

# Level 3: 全局 Control 兜底
ctrl_lvl3 = train_control_log2.mean(skipna=True)

# Level 4: Baseline 1 全局蛋白均值（极极端兜底）
ctrl_lvl4 = protein_mean_vector  # 来自 Baseline 1

print(f"Level 1 组合数 (strain×medium×temp): {len(ctrl_lvl1)}")
print(f"Level 2 组合数 (strain): {len(ctrl_lvl2)}")
print(f"Level 3 全局 Control 向量长度: {len(ctrl_lvl3)}")
print(f"Level 4 = Baseline 1 全局均值向量长度: {len(ctrl_lvl4)}")

Level 1 组合数 (strain×medium×temp): 16
Level 2 组合数 (strain): 4
Level 3 全局 Control 向量长度: 4422
Level 4 = Baseline 1 全局均值向量长度: 4422


In [9]:
# 3. 遍历验证集样本，按逐级回退策略构建 Control 预测矩阵
val_meta = meta.loc[val_mask]
n_val = val_log2.shape[0]
n_proteins = val_log2.shape[1]

val_preds_control = pd.DataFrame(
    np.full((n_val, n_proteins), np.nan),
    index=val_log2.index,
    columns=val_log2.columns
)

level_counts = {1: 0, 2: 0, 3: 0, 4: 0}

for i, (sid, row) in enumerate(val_meta.iterrows()):
    strain = row['Strains']
    medium = row['Medium']
    temp = row['Temperature']
    pred_vec = None
    used_level = None

    # Level 1: strain + medium + temperature 严格匹配
    if (strain, medium, temp) in ctrl_lvl1.index:
        pred_vec = ctrl_lvl1.loc[(strain, medium, temp)].values
        used_level = 1

    # Level 2: 同菌株回退
    elif strain in ctrl_lvl2.index:
        pred_vec = ctrl_lvl2.loc[strain].values
        used_level = 2

    # Level 3: 全局 Control 兜底
    elif len(train_control_log2) > 0:
        pred_vec = ctrl_lvl3.values
        used_level = 3

    # Level 4: 极极端兜底 - Baseline 1 全局均值
    else:
        pred_vec = ctrl_lvl4.values
        used_level = 4

    val_preds_control.iloc[i] = pred_vec
    level_counts[used_level] += 1

# 填补预测矩阵中残留的 NaN（某些蛋白在对应 Control 组中全缺失）
# 依次用 Level 3 全局 Control 均值、Level 4 Baseline1 均值填充
val_preds_control = val_preds_control.fillna(ctrl_lvl3)
val_preds_control = val_preds_control.fillna(ctrl_lvl4)

print("===== 验证集 Control 匹配统计 =====")
for lvl in [1, 2, 3, 4]:
    cnt = level_counts[lvl]
    pct = cnt / n_val * 100
    print(f"Level {lvl}: {cnt} 样本 ({pct:.2f}%)")


===== 验证集 Control 匹配统计 =====
Level 1: 1222 样本 (40.22%)
Level 2: 0 样本 (0.00%)
Level 3: 1816 样本 (59.78%)
Level 4: 0 样本 (0.00%)


In [10]:
# 4. 评估指标计算与对比
# 只针对真实值中非 NaN 的有效位置进行评估
valid_mask_ctrl = val_true.notna().values

y_true_ctrl = val_true.values[valid_mask_ctrl]
y_pred_ctrl = val_preds_control.values[valid_mask_ctrl]

# Baseline 2 指标
rmse_ctrl = np.sqrt(np.mean((y_true_ctrl - y_pred_ctrl) ** 2))
r2_ctrl = r2_score(y_true_ctrl, y_pred_ctrl)

print("===== Baseline 2 (Control Baseline + 逐级回退) 结果 =====")
print(f"对齐后总样本数: {len(common_ids)}")
print(f"保留的蛋白列数量: {len(valid_protein_cols)}")
print(f"验证集样本数: {val_log2.shape[0]}")
print(f"有效评估位置数 (非缺失): {valid_mask_ctrl.sum()}")
print(f"RMSE: {rmse_ctrl:.6f}")
print(f"R2 Score: {r2_ctrl:.6f}")

print("\n" + "=" * 60)
print("===== Baseline 1 vs Baseline 2 对比 =====")
print(f"{'指标':<20} {'Baseline 1':<18} {'Baseline 2':<18} {'变化':<18}")
print("-" * 60)
rmse_diff = rmse_ctrl - rmse
r2_diff = r2_ctrl - r2
rmse_pct = (rmse_ctrl - rmse) / rmse * 100
r2_pct = (r2_ctrl - r2) / abs(r2) * 100 if r2 != 0 else float('nan')
print(f"{'RMSE':<20} {rmse:<18.6f} {rmse_ctrl:<18.6f} {rmse_diff:+.6f} ({rmse_pct:+.2f}%)")
print(f"{'R2 Score':<20} {r2:<18.6f} {r2_ctrl:<18.6f} {r2_diff:+.6f} ({r2_pct:+.2f}%)")
print("-" * 60)
print(f"RMSE 降低: {abs(rmse_diff):.6f} ({abs(rmse_pct):.2f}% 改善)" if rmse_diff < 0 else f"RMSE 升高: {rmse_diff:.6f}")
print(f"R2 提升: {r2_diff:+.6f}" if r2_diff > 0 else f"R2 下降: {r2_diff:+.6f}")

===== Baseline 2 (Control Baseline + 逐级回退) 结果 =====
对齐后总样本数: 13412
保留的蛋白列数量: 4422
验证集样本数: 3038
有效评估位置数 (非缺失): 11521736
RMSE: 0.905062
R2 Score: 0.893648

===== Baseline 1 vs Baseline 2 对比 =====
指标                   Baseline 1         Baseline 2         变化                
------------------------------------------------------------
RMSE                 0.938526           0.905062           -0.033464 (-3.57%)
R2 Score             0.885639           0.893648           +0.008010 (+0.90%)
------------------------------------------------------------
RMSE 降低: 0.033464 (3.57% 改善)
R2 提升: +0.008010
